# Trajectory Distribution Fuzzing in Highway Roundabout

This notebook demonstrates trajectory distribution fuzzing for the HighwayEnv roundabout environment.

**Key Components** (now in `src/` modules):
- `ScenarioParams`: Dataclass for configurable observation, action, and environment disturbances
- `SimulatedEnv`: Custom roundabout environment with fuzzable parameters
- `rollout()`: Runs episodes with noise injection and computes log-probabilities
- `ScenarioFuzzer`: Class for finding critical failure scenarios via optimization

**Usage**: Create a `ScenarioParams` object and pass it to `gym.make("SimulatedEnv-v0", scenario_params=...)`

In [ ]:
# Uncomment to install dependencies
# !pip install gymnasium stable_baselines3 opencv-python highway_env torch

In [4]:
import gymnasium as gym
from gymnasium.wrappers import RecordVideo
from stable_baselines3 import DQN
import numpy as np

# Import from src modules
from src import (
    ScenarioParams,
    GaussianMixtureParam,
    NormalParam,
    ProbabilityParam,
    BetaParam,
    NOMINAL,
    SQRT_2,
    register_env,
    rollout,
    ScenarioFuzzer,
    FuzzerConfig,
    PARAM_NAMES,
    compute_robustness,
    weights_from_vector,
)

In [ ]:
# Register the SimulatedEnv with gymnasium
register_env()

In [ ]:
# Define scenario parameters (customize as needed)
setup = ScenarioParams(
    # Observation disturbances
    initial_position_x=GaussianMixtureParam(
        p=[1.0], 
        mu=[0.0, 0.0], 
        sigma=[0.005, 0.005]
    ),
    initial_position_y=GaussianMixtureParam(
        p=[1.0], 
        mu=[0.0, 0.0], 
        sigma=[0.005, 0.005]
    ),
    velocity_x=GaussianMixtureParam(
        p=[1.0],
        mu=[0.0, 0.0],
        sigma=[SQRT_2 * 0.005 / 0.1, SQRT_2 * 0.005 / 0.1]
    ),
    velocity_y=GaussianMixtureParam(
        p=[1.0],
        mu=[0.0, 0.0],
        sigma=[SQRT_2 * 0.005 / 0.1, SQRT_2 * 0.005 / 0.1]
    ),
    # Action disturbances
    high_lvl_ctrl_noise=ProbabilityParam(p=[0.0]),
    initial_speed=NormalParam(mu=8.0, sigma=0.5),
    initial_heading=ProbabilityParam(p=[0.33, 0.33, 0.33]),
    # Environment disturbances
    politeness=BetaParam(ab=[1, 1]),  # uniform distribution
    other_vehicle_speed=NormalParam(mu=16.0, sigma=2.0),
    entering_vehicle_position=GaussianMixtureParam(
        p=[1.0],
        mu=[5.0, 5.0],  # tip: mu=100.0 can cause more crashes
        sigma=[2.0, 2.0]
    )
)

In [ ]:
TRAIN = False

# Create the environment
env = gym.make("SimulatedEnv-v0", render_mode="rgb_array", scenario_params=setup)
obs, info = env.reset()

# Create the model
model = DQN(
    "MlpPolicy",
    env,
    policy_kwargs=dict(net_arch=[256, 256]),
    learning_rate=5e-4,
    buffer_size=15000,
    learning_starts=200,
    batch_size=32,
    gamma=0.8,
    train_freq=1,
    gradient_steps=1,
    target_update_interval=50,
    verbose=1,
)

In [ ]:
# Train the model (if needed)
if TRAIN:
    model.learn(total_timesteps=int(2e4))
    model.save("roundabout_dqn/model")
    del model

In [ ]:
# Load trained model
model = DQN.load("roundabout_dqn/model.zip", env=env, device="cpu")

# Set up video recording
env = RecordVideo(
    env, video_folder="roundabout_dqn/videos/", episode_trigger=lambda e: True
)
env.unwrapped.config["simulation_frequency"] = 15
env.unwrapped.config.update({
    "observation": {
        "type": "Kinematics",
        "absolute": True,
        "features_range": {
            "x": [-100, 100],
            "y": [-100, 100],
            "vx": [-15, 15],
            "vy": [-15, 15],
        },
        "vehicles_count": 5,
    },
})
env.unwrapped.set_record_video_wrapper(env)

In [ ]:
# Run rollouts and collect log-probabilities + robustness scores
robustness_weights = weights_from_vector([0.2042, 0.1982, 0.4393, 0.0111, 0.1471])

logs = []
for _ in range(2):
    result = rollout(env, model, robustness_weights=robustness_weights, eps=1e-10)
    logs.append(result)

print(logs)

## Fuzzing for Critical Failures

Use `ScenarioFuzzer` to find scenario parameters that minimize robustness while keeping scenarios plausible.

In [ ]:
# Create fuzzer with custom config
config = FuzzerConfig(
    n_samples=3,           # rollouts per evaluation
    log_prob_weight=0.01,  # penalty for implausible scenarios
    verbose=True,
)

fuzzer = ScenarioFuzzer(
    env=env,
    model=model,
    scenario_params=setup,
    robustness_weights=robustness_weights,
    config=config,
)

# Show parameter bounds
print("Parameter bounds:")
for name, (lo, hi), val in zip(PARAM_NAMES, config.bounds, config.initial_guess):
    print(f"  {name}: [{lo}, {hi}], initial={val}")

In [ ]:
# Run optimization (WARNING: this can take a while)
# Uncomment to run

# result = fuzzer.optimize(method="L-BFGS-B", maxiter=50)
# print(f"\nOptimization finished: {result.message}")
# print(f"Critical failure parameters:")
# for name, val in fuzzer.get_params_dict(result.x).items():
#     print(f"  {name}: {val:.4f}")
# print(f"\nBest result: {fuzzer.best_result}")

In [ ]:
env.close()